# 04 — Evaluate Climatology-Matched (`clim`) and Observationally Matched (`obs`) ICs Against Actual 2015

**One branch per run**: set `EVAL_BRANCH` in the configuration cell to `"clim"`, `"obs"` (ERA5), or `"obs_era5land_nsidc"` (ERA5-Land + NOAA/NSIDC). Figures and tables go to `DIAG_ROOT/<EVAL_BRANCH>/`.


This notebook evaluates how closely the selected candidate years from the `clim` and `obs*` branches (`obs`: ERA5; `obs_era5land_nsidc`: ERA5-Land land + NOAA/NSIDC CDR sea ice) match the **actual 2015 BruteForce restarts** across the four sensitivity-test initialization dates (`2015-02-01`, `2015-05-01`, `2015-08-01`, `2015-11-01`).

All evaluation functions are implemented in [`util/ic_eval.py`](util/ic_eval.py), and all tables and figures are saved to `/global/cfs/cdirs/e3sm/www/zhan391/E3SMv3_S2D/rtenso2026_ic/`.

### Contents
1. **Configuration & selection summary** (selected years per branch)
2. **Distance & rank evaluation against actual 2015** (`evaluate_distance_ranks`, `plot_distance_vs_truth`)
3. **Field-level similarity metrics (`d_box`, `r_box`, `d_grid`, `r_grid`, `RMSE`, `bias`)** (`field_similarity_table`, `plot_field_similarity_bars`)
4. **Global & hemispheric integrated restart summary** (`global_summary_table`)
5. **Large-scale (`BOX`) standardized anomaly maps** (`plot_box_spatial`)

In [ ]:
from pathlib import Path
import sys
import IPython.display as ipd
import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, str(Path("util").resolve()))
import importlib
import clim_restart
import ic_eval as ice

# Reload so edits to util/ take effect without restarting the kernel.
importlib.reload(clim_restart)
ice = importlib.reload(ice)

ice.ensure_proj_data()
plt.rcParams["figure.dpi"] = 120

# Branch evaluated in this run: "clim", "obs" (ERA5), or "obs_era5land_nsidc".
# Figures and tables go to DIAG_ROOT/<EVAL_BRANCH>/.
EVAL_BRANCH = "obs_era5land_nsidc"

SRC_ROOT   = Path("/global/cfs/cdirs/e3sm/zhan391/v3.LR.S2D.ENSINT/BruteForce")
# Observation sets for the obs branches (as in 02_build_climatology_ic.ipynb);
# "label" names each set's observed anomaly in the tables and maps.
ERA5_DIR = Path("/global/cfs/projectdirs/m3522/cmip6/ERA5/e5.oper.an.sfc")
OBS_DATA = Path("/global/cfs/cdirs/e3sm/zhan391/data")
OBS_SETS = {
    "obs": {
        "label": "ERA5",
        "root": {"elm": ERA5_DIR, "mpassi": ERA5_DIR},
        "fields": {"elm": ["sd", "swvl1", "swvl2-4", "stl1", "stl2-4"], "mpassi": ["ci"]},
    },
    "obs_era5land_nsidc": {
        "label": "ERA5-Land/NSIDC",
        "root": {"elm": OBS_DATA / "ERA5_LND", "mpassi": OBS_DATA / "NOAA_NSIDC"},
        "fields": {"elm": ["sd", "swvl1", "swvl2-4", "stl1", "stl2-4"], "mpassi": ["ci"]},
    },
}
BRANCHES = [EVAL_BRANCH]
OUT_ROOT   = Path("/global/cfs/cdirs/e3sm/S2S2D/RTENSO2026/initial_conditions")
CACHE_DIR  = OUT_ROOT / "cache"
DIAG_ROOT  = Path("/global/cfs/cdirs/e3sm/www/zhan391/E3SMv3_S2D/rtenso2026_ic") / EVAL_BRANCH
SAVE_DIAG  = True

TEST_DATES   = ["2015-02-01", "2015-05-01", "2015-08-01", "2015-11-01"]
EXAMPLE_DATE = "2015-02-01"
BOX          = (15.0, 30.0)   # as in 02 (from 01_choose_box_size.ipynb)
BOX_SHIFTS   = 4             # as in 02 (selection distance only)
FIRST_YEAR   = 1995          # candidate years FIRST_YEAR-2018, as in 02 (None = 1980)

plans, distances, selections = ice.load_test_selections(
    TEST_DATES, SRC_ROOT, OUT_ROOT, obs_sets=OBS_SETS, branches=BRANCHES, first_year=FIRST_YEAR,
    box=BOX, box_shifts=BOX_SHIFTS, cache_dir=CACHE_DIR
)


def branch_years(date, comp="mpassi"):
    return {b: selections[b, date][comp] for b in BRANCHES if (b, date) in selections}


def obs_plans(date, key="label"):
    """Plans of the obs sets, keyed by their label, for the observed-anomaly references."""
    return {OBS_SETS[b][key] if key else b: plans[b, date] for b in OBS_SETS if (b, date) in plans}


for td in TEST_DATES:
    print(f"{td}: " + "  ".join(f"{b}={y}" for b, y in branch_years(td).items()))

## 1. Distance and Rank Against Actual 2015

Each candidate year $y \in [\mathrm{FIRST\_YEAR}, 2018] \setminus \{2015\}$ is scored directly against the actual 2015 BruteForce restart on `BOX` boxes (`D_true`, Rank 1 = closest to actual 2015 among the candidate years (23 for `FIRST_YEAR = 1995`)).

In [ ]:
# --- Figure setup ---
FONTSIZE = 14        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG

rank_summary_df, truth_obs, truth_all = ice.evaluate_distance_ranks(
    TEST_DATES, plans, distances, box=BOX, cache_dir=CACHE_DIR
)
ice.save_table(rank_summary_df, "2015_branch_rank_summary", DIAG_ROOT, SAVE_DIAG)
ipd.display(rank_summary_df.round(3))

fig = ice.plot_distance_vs_truth(TEST_DATES, distances, truth_obs, fontsize=FONTSIZE, panel_size=PANEL_SIZE)
ice.save_fig(fig, "2015_distance_vs_truth_scatter", DIAG_ROOT, SAVE_DIAG, dpi=DPI)
plt.show()

## 2. Field-Level Similarity Metrics (`d_box`, `r_box`, `d_grid`, `r_grid`, `RMSE`, `bias`)

For every key sea-ice and land field, we compare each branch's selected year and each observation set's anomaly (`ERA5`, `ERA5-Land/NSIDC`) against the actual 2015 state on both `BOX` boxes and the native model grid.

In [ ]:
# --- Figure setup ---
FONTSIZE = 14        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG

sim_all = pd.concat([
    ice.field_similarity_table(
        plans[EVAL_BRANCH, td], 2015,
        branch_years(td), obs_plans=obs_plans(td),
        box=BOX, cache_dir=CACHE_DIR, log=lambda *_: None,
    )
    for td in TEST_DATES
], ignore_index=True)

ice.save_table(sim_all, "2015_field_similarity_all_dates", DIAG_ROOT, SAVE_DIAG)
ipd.display(sim_all[sim_all["target_date"] == EXAMPLE_DATE].round(3))

mean_by_source = (
    sim_all.groupby(["component", "field", "source"])[["d_box", "r_box", "d_grid", "r_grid", "RMSE", "PCC", "bias"]]
    .mean()
    .round(3)
)
ice.save_table(mean_by_source.reset_index(), "2015_field_similarity_4season_mean", DIAG_ROOT, SAVE_DIAG)
ipd.display(mean_by_source)

fig = ice.plot_field_similarity_bars(sim_all, fontsize=FONTSIZE, panel_size=PANEL_SIZE)
ice.save_fig(fig, "2015_field_similarity_bars", DIAG_ROOT, SAVE_DIAG, dpi=DPI)
plt.show()

## 3. Global & Hemispheric Integrated Restart Summary (`Actual 2015` vs each branch)

In [ ]:
summary_2015_df = ice.global_summary_table(TEST_DATES, selections, SRC_ROOT)
ice.save_table(summary_2015_df, "2015_global_integrated_summary", DIAG_ROOT, SAVE_DIAG)
ipd.display(summary_2015_df.round(3))

## 4. `BOX`-Scale Standardized Anomaly Maps

Since candidate years are selected using `BOX`-averaged standardized anomalies, these maps show what the selection metric directly sees.

Columns: the truth (observed anomaly), the reference (actual 2015 E3SM state), and the selected initial
condition; box PCC and RMSE compare the second and third with the observed anomaly.


In [ ]:
# --- Figure setup ---
FONTSIZE = 14        # panel-title font size; titles, labels, and ticks scale from it
PANEL_SIZE = None    # (width, height) of each panel in inches; None scales with FONTSIZE
DPI = 150            # resolution of the saved PNG
Z_LIM = 2.0          # color range of the box anomaly maps

plan_ex = plans[EVAL_BRANCH, EXAMPLE_DATE]
byears_ex = branch_years(EXAMPLE_DATE)

box_fields = [
    ("mpassi", "NH ice concentration"),
    ("mpassi", "SH ice concentration"),
    ("elm", "snow water equivalent"),
    ("elm", "surface soil water"),
    ("elm", "subsurface soil water"),
    ("elm", "surface soil temperature"),
]

for comp, fname in box_fields:
    sp = ice.prepare_spatial_field(
        plan_ex, comp, fname, 2015, byears_ex, obs_plans=obs_plans(EXAMPLE_DATE), box=BOX, cache_dir=CACHE_DIR, log=lambda *_: None
    )
    fig = ice.plot_box_spatial(sp, z_lim=Z_LIM, fontsize=FONTSIZE, panel_size=PANEL_SIZE)
    ice.save_fig(
        fig, f"box{BOX[0]:g}x{BOX[1]:g}_z_{comp}_{ice.slugify(fname)}_{EXAMPLE_DATE.replace('-', '')}",
        DIAG_ROOT, SAVE_DIAG, dpi=DPI,
    )
    plt.show()

## 5. Key findings

Evaluated branch: `obs_era5land_nsidc` (`EVAL_BRANCH`), selection with `BOX = (15, 30)`, `BOX_SHIFTS = 4`.

**Table. Selected year and its closeness to the actual 2015 state.** Rank among the 23 candidate years
(1 = closest to actual 2015) using the observed fields and all key fields (adding sea-ice thickness and snow on
ice); rank correlation between the observation-based distance and the true distance over all candidates.

| Date | Selected year | Rank (observed fields) | Rank (all key fields) | Rank correlation |
|---|---|---|---|---|
| 2015-02-01 | 2014 | 1 | 1 | 0.85 |
| 2015-05-01 | 2014 | 1 | 1 | 0.91 |
| 2015-08-01 | 2017 | 1 | 7 (closest: 2009) | 0.94 |
| 2015-11-01 | 2014 | 1 | 1 | 0.71 |

1. **The observation-based selection finds the closest candidate year.** On all four 2015 dates the selected
   year is the candidate closest to the actual 2015 state on the observed fields, and the observation-based
   distance ranks all candidates consistently with the true distance (rank correlation 0.71-0.94). In Aug the
   unobserved sea-ice thickness and snow on ice favour a different year (2009).
2. **The closest year still matches the actual 2015 anomaly only partly.** Over the four dates (box scale), the
   selected year agrees well with actual 2015 for snow water equivalent (pattern correlation r_box = 0.75) and
   moderately for NH ice thickness (0.53) and subsurface soil temperature (0.44), but weakly for surface soil
   temperature (0.14), soil water (0.15-0.39), and sea-ice concentration (0.34 NH, 0.47 SH).
3. **The observed 2015 anomaly itself matches the actual 2015 anomaly much better** for surface soil temperature
   (r_box = 0.86), subsurface soil temperature (0.74), surface soil water (0.68), and sea-ice concentration
   (0.77 NH, 0.85 SH). These anomalies are
   observable but no archive year reproduces them, which is what `clim_anom` (`03_build_clim_anomaly_ic.ipynb`)
   addresses.
4. **Integrated totals are close to actual 2015.** Global land snow, soil water, soil temperature, and water
   table are within 0-4%, and river storage within 1-4%. NH sea-ice area and extent are within 4% except Aug
   (area -10%, volume -21%). SH sea ice is furthest off in Feb (area -14%, extent -19%, snow volume -40%) and
   within about 6% otherwise.
5. **Box anomaly maps (section 4)** compare, against the observed anomaly (the truth), the actual 2015 model state
   and the selected initial condition on 2015-02-01. For surface soil temperature the actual model state
   matches the observed anomaly well (box PCC = 0.85) but the selected year (2014) does not (box PCC = -0.01): the
   observed soil-temperature anomaly is not reproduced by any archive year.
